In [24]:
import requests
import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────────
USERNAME = "marclambertsmedia@gmail.com"
PASSWORD = "Meneertosti@1!"
BASE_URL = "https://api.impect.com/v5/customerapi"
AUTH_URL = "https://login.impect.com/auth/realms/production/protocol/openid-connect/token"

def unwrap(resp) -> list:
    resp.raise_for_status()
    return resp.json()["data"]

# Auth
resp = requests.post(AUTH_URL, data={
    "grant_type": "password",
    "client_id":  "api",
    "username":   USERNAME,
    "password":   PASSWORD,
})
resp.raise_for_status()
HEADERS = {"Authorization": f"Bearer {resp.json()['access_token']}"}
print("✓ Authenticated")

# ── Set iteration ─────────────────────────────────────────────────────────────
ITERATION_ID = 1454   # ← change: 1421=Czech top, 1432=Slovak top, 1454=Czech 2nd

# Squad lookup
squads_raw  = unwrap(requests.get(f"{BASE_URL}/iterations/{ITERATION_ID}/squads", headers=HEADERS))
squad_names = {s["id"]: s["name"] for s in squads_raw}
print(f"✓ {len(squad_names)} squads in iteration {ITERATION_ID}")

# Player identity lookup
players_raw = unwrap(requests.get(f"{BASE_URL}/iterations/{ITERATION_ID}/players", headers=HEADERS))
players_info = pd.json_normalize(players_raw)[["id", "firstname", "lastname", "commonname", "birthdate", "leg"]]
players_info = players_info.rename(columns={"id": "playerId"})
print(f"✓ {len(players_info)} players in iteration {ITERATION_ID}")

✓ Authenticated
✓ 16 squads in iteration 1454
✓ 610 players in iteration 1454


In [25]:
# ── Squad KPIs + Scores → one Excel ──────────────────────────────────────────
squad_kpis_df = pd.json_normalize(
    unwrap(requests.get(f"{BASE_URL}/iterations/{ITERATION_ID}/squad-kpis", headers=HEADERS))
)
squad_scores_df = pd.json_normalize(
    unwrap(requests.get(f"{BASE_URL}/iterations/{ITERATION_ID}/squad-scores", headers=HEADERS))
)

# Merge on squadId, add name
squad_df = squad_kpis_df.merge(squad_scores_df, on="squadId", suffixes=("_kpi", "_score"))
squad_df.insert(0, "squadName", squad_df["squadId"].map(squad_names))

out = f"squads_{ITERATION_ID}.xlsx"
squad_df.to_excel(out, index=False)
print(f"✓ Squads: {squad_df.shape}  → {out}")

✓ Squads: (16, 6)  → squads_1454.xlsx


In [ ]:
# ── Player KPIs + Scores + Profile/Position Scores → one row per player ───────
ALL_POSITIONS = ",".join([
    "GOALKEEPER", "CENTRAL_DEFENDER", "RIGHT_WINGBACK_DEFENDER", "LEFT_WINGBACK_DEFENDER",
    "DEFENSE_MIDFIELD", "CENTRAL_MIDFIELD", "RIGHT_WINGER", "LEFT_WINGER",
    "ATTACKING_MIDFIELD", "CENTER_FORWARD"
])

# Fetch name lookup tables
kpi_names   = {k["id"]: k["name"] for k in unwrap(requests.get(f"{BASE_URL}/kpis",           headers=HEADERS))}
score_names = {s["id"]: s["name"] for s in unwrap(requests.get(f"{BASE_URL}/player-scores",   headers=HEADERS))}

def expand_nested(raw, id_col, val_col, names):
    """Explode list-of-{id,value} dicts into wide columns per player."""
    rows = []
    for rec in raw:
        player_id = rec["playerId"]
        for item in rec.get(val_col, []):
            rows.append({
                "playerId": player_id,
                "col":  names.get(item[id_col], f"{id_col}_{item[id_col]}"),
                "value": item["value"],
            })
    return (
        pd.DataFrame(rows)
        .groupby(["playerId", "col"])["value"].mean()
        .unstack("col")
        .reset_index()
    )

all_kpis        = []
all_scores      = []
all_profile     = []
all_pos_scores  = []
play_duration   = []   # collect playDuration / matchShare from any endpoint

for squad_id, squad_name in squad_names.items():
    base = f"{BASE_URL}/iterations/{ITERATION_ID}/squads/{squad_id}"

    # player-kpis
    try:
        raw = unwrap(requests.get(f"{base}/player-kpis", headers=HEADERS))
        play_duration.append(
            pd.DataFrame([{"playerId": r["playerId"], "squadId": squad_id, "squadName": squad_name,
                           "playDuration": r["playDuration"], "matchShare": r["matchShare"]} for r in raw])
        )
        df = expand_nested(raw, "kpiId", "kpis", kpi_names)
        df["squadId"] = squad_id
        all_kpis.append(df)
    except Exception as e:
        print(f"  ✗ player-kpis {squad_name}: {e}")

    # player-scores
    try:
        raw = unwrap(requests.get(f"{base}/player-scores", headers=HEADERS))
        df = expand_nested(raw, "playerScoreId", "playerScores", score_names)
        df["squadId"] = squad_id
        all_scores.append(df)
    except Exception as e:
        print(f"  ✗ player-scores {squad_name}: {e}")

    # player-profile-scores
    try:
        raw = unwrap(requests.get(f"{base}/positions/{ALL_POSITIONS}/player-profile-scores", headers=HEADERS))
        df = expand_nested(raw, "playerScoreId", "profileScores", score_names)
        df["squadId"] = squad_id
        all_profile.append(df)
    except Exception as e:
        print(f"  ✗ profile-scores {squad_name}: {e}")

    # position player-scores
    try:
        raw = unwrap(requests.get(f"{base}/positions/{ALL_POSITIONS}/player-scores", headers=HEADERS))
        df = expand_nested(raw, "playerScoreId", "playerScores", score_names)
        df["squadId"] = squad_id
        all_pos_scores.append(df)
    except Exception as e:
        print(f"  ✗ pos-scores {squad_name}: {e}")

# Build base: play duration + squad info (one row per player, last squad wins)
dur_df = pd.concat(play_duration, ignore_index=True)
base_df = (
    dur_df.groupby("playerId", as_index=False)
    .agg(playDuration=("playDuration","sum"), matchShare=("matchShare","sum"),
         squadId=("squadId","last"), squadName=("squadName","last"))
)

# Average each metric group per player then merge
def avg_merge(frames):
    df = pd.concat(frames, ignore_index=True)
    num = [c for c in df.columns if c not in ("playerId","squadId")]
    return df.groupby("playerId", as_index=False)[num].mean()

kpis_wide   = avg_merge(all_kpis)
scores_wide = avg_merge(all_scores)

players_df = base_df.merge(kpis_wide,   on="playerId", how="left", suffixes=("","_dup"))
players_df = players_df.merge(scores_wide, on="playerId", how="left", suffixes=("","_dup"))
players_df.drop(columns=[c for c in players_df.columns if c.endswith("_dup")], inplace=True)

if all_profile:
    profile_wide = avg_merge(all_profile)
    players_df = players_df.merge(profile_wide, on="playerId", how="left", suffixes=("","_profile"))

if all_pos_scores:
    pos_wide = avg_merge(all_pos_scores)
    players_df = players_df.merge(pos_wide, on="playerId", how="left", suffixes=("","_pos"))

# Add player identity
players_df = players_df.merge(players_info, on="playerId", how="left")

# Identity columns first
front = ["playerId","firstname","lastname","commonname","birthdate","leg","squadName","squadId","playDuration","matchShare"]
front = [c for c in front if c in players_df.columns]
rest  = [c for c in players_df.columns if c not in front]
players_df = players_df[front + rest]

out = f"players_{ITERATION_ID}.xlsx"
players_df.to_excel(out, index=False)
print(f"✓ {len(players_df)} unique players, {len(players_df.columns)} columns → {out}")
display(players_df[["playerId","firstname","lastname","commonname","squadName","playDuration"]].head(10))

In [ ]:
# ── Player Profile Scores & Position Scores — per squad + position ────────────
# Available positions:
# GOALKEEPER, CENTRAL_DEFENDER, RIGHT_WINGBACK_DEFENDER, LEFT_WINGBACK_DEFENDER,
# DEFENSE_MIDFIELD, CENTRAL_MIDFIELD, RIGHT_WINGER, LEFT_WINGER,
# ATTACKING_MIDFIELD, CENTER_FORWARD

SQUAD_ID  = list(squad_names.keys())[0]
POSITIONS = "CENTER_FORWARD,LEFT_WINGER,RIGHT_WINGER"

profile_scores_df = pd.json_normalize(unwrap(requests.get(
    f"{BASE_URL}/iterations/{ITERATION_ID}/squads/{SQUAD_ID}/positions/{POSITIONS}/player-profile-scores",
    headers=HEADERS
)))
profile_scores_df["squadName"] = squad_names.get(SQUAD_ID)

position_scores_df = pd.json_normalize(unwrap(requests.get(
    f"{BASE_URL}/iterations/{ITERATION_ID}/squads/{SQUAD_ID}/positions/{POSITIONS}/player-scores",
    headers=HEADERS
)))
position_scores_df["squadName"] = squad_names.get(SQUAD_ID)

safe_pos = POSITIONS.replace(",", "_")
profile_scores_df.to_excel(f"player_profile_scores_{SQUAD_ID}_{safe_pos}.xlsx", index=False)
position_scores_df.to_excel(f"player_position_scores_{SQUAD_ID}_{safe_pos}.xlsx", index=False)

print(f"✓ Profile Scores ({POSITIONS}): {profile_scores_df.shape}  → player_profile_scores_{SQUAD_ID}_{safe_pos}.xlsx")
print(f"✓ Position Scores ({POSITIONS}): {position_scores_df.shape}  → player_position_scores_{SQUAD_ID}_{safe_pos}.xlsx")